# Program 5: Improve a prompt with embedding-retrieved words

**Aim.** Retrieve similar words with embeddings and use them to enrich a GenAI prompt.

**Theory.** Prompt enrichment adds relevant terms so the model covers more of what the user means. Embeddings supply candidate terms; code filters them so the prompt does not fill with noise.

In [1]:
import sys
sys.path.insert(0, '.')
from common import *


In [2]:
import gensim.downloader as api
wv = api.load('glove-wiki-gigaword-100')
STOP = set('the a an of and or to in is for on with'.split())

def related(seed, n=6):
    out = []
    for w, s in wv.most_similar(seed, topn=40):
        if w.isalpha() and w not in STOP and seed not in w and w not in out and len(w) > 3:
            out.append(w)
        if len(out) == n:
            break
    return out

seeds = ['medicine', 'dose', 'pharmacist']
extra = {s: related(s) for s in seeds}
for s, ws in extra.items():
    print(f'{s:10s} ->', ', '.join(ws))

medicine   -> medical, health, veterinary, nutrition, studies, physicians
dose       -> dosage, injections, medication, placebo, chemotherapy, pills
pharmacist -> dentist, physician, internist, pediatrician, anesthesiologist, urologist


In [3]:
base = 'Explain how a patient should take a new medicine.'
terms = []
for ws in extra.values():
    terms += [w for w in ws[:3] if w not in terms]
enriched = f"{base} Cover these related ideas where they matter: {', '.join(terms)}."
print('ORIGINAL :', base)
print('ENRICHED :', enriched)
print('added', len(terms), 'terms')
import json; (OUT / 'lab05_prompts.json').write_text(json.dumps({'original': base, 'enriched': enriched, 'terms': terms}), encoding='utf-8')

ORIGINAL : Explain how a patient should take a new medicine.
ENRICHED : Explain how a patient should take a new medicine. Cover these related ideas where they matter: medical, health, veterinary, dosage, injections, medication, dentist, physician, internist.
added 9 terms


388

## Observations

The retrieved words are plausible but not all useful (some are generic). Filtering by length, alphabet and duplicates keeps the prompt clean, and capping the count keeps it short. Whether enrichment helps is tested in Program 6.